# 01. Limpieza de datos y metodología CRISP-DM
## Proyecto final
### Modelo predictivo Mundial 2026

**Responsable de esta sección (Persona 2):** limpieza de datos, selección de variables, y redacción de
*Business Understanding*, *Data Understanding* y *Data Preparation*.

Este notebook toma los archivos crudos descargados por Persona 1 (documentados en
`documentacion_fuentes_mundial_2026.md`) y produce el dataset limpio que usará Persona 3 para entrenar
el modelo predictivo (`02_modelo_predictivo.ipynb`).

**Salidas de este notebook:**

- `data/processed/clean_model_dataset.csv`: dataset a nivel de partido, con variables de forma
  reciente, fuerza relativa y contexto del partido, listo para modelar.
- `data/processed/team_features_2026.csv`: foto actual (ranking FIFA, Elo, forma reciente) de cada
  selección, usada como punto de partida para simular el Mundial 2026 hacia adelante.


## 1. Business Understanding

**Objetivo del negocio/proyecto:** construir, de forma reproducible y justificable con datos, un
modelo que estime la probabilidad de que cada selección participante en el Mundial 2026 llegue a
semifinales, final, y finalmente prediga el podio completo: campeón, subcampeón, 3° y 4° lugar.

**Por qué importa:** la evaluación premia el rigor metodológico (trazabilidad de los datos, decisiones
de limpieza justificadas, y un modelo explicable) por encima de acertar el resultado final. El Mundial
2026 se encuentra en fase eliminatoria y la final es el 19 de julio, por lo que el modelo debe poder
generar una predicción de podio *antes* de esa fecha, usando información disponible hasta el momento
del corte de datos (8 de julio de 2026).

**Criterios de éxito:**

1. El dataset limpio debe permitir entrenar un modelo de resultado de partido (o de fuerza relativa
   entre selecciones) sin fugas de información (*data leakage*) hacia el futuro.
2. Cada variable incluida debe estar justificada: de dónde sale, qué mide, y por qué es relevante para
   predecir desempeño en un torneo eliminatorio.
3. El dataset debe ser reutilizable tanto para entrenar (histórico) como para simular el Mundial 2026
   hacia adelante (foto actual de cada selección).



## 2. Data Understanding

Fuentes utilizadas (ver detalle completo en `documentacion_fuentes_mundial_2026.md`):

| Archivo | Descripción | Nivel |
|---|---|---|
| `results.csv` | Historial de partidos internacionales (Kaggle, M. Jürisoo) | partido |
| `shootouts.csv` | Resultados de tandas de penales | partido |
| `fifa_ranking_men_2026_07_08.csv` | Ranking oficial FIFA, foto 2026-07-08 | selección |
| `elo_ratings_2026_07_08.csv` | Rating Elo, foto 2026-07-08 | selección |

Los archivos `goalscorers.csv` y los de Transfermarkt (`opcional/`) quedan documentados pero **no se
usan** en la primera versión del modelo, siguiendo la recomendación de Persona 1 de mantener el dataset
base simple y justificable.

A continuación se cargan los archivos crudos y se hace una inspección inicial: dimensiones, tipos de
dato, valores nulos, cobertura de fechas y consistencia de nombres de selecciones entre fuentes.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from unidecode import unidecode

pd.set_option("display.max_columns", 50)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(RAW_DIR / "results.csv", parse_dates=["date"])
shootouts = pd.read_csv(RAW_DIR / "shootouts.csv", parse_dates=["date"])
fifa_rank = pd.read_csv(RAW_DIR / "fifa_ranking_men_2026_07_08.csv")
elo = pd.read_csv(RAW_DIR / "elo_ratings_2026_07_08.csv")

print("results:", results.shape)
print("shootouts:", shootouts.shape)
print("fifa_rank:", fifa_rank.shape)
print("elo:", elo.shape)


In [ ]:
# Inspección de tipos y nulos por dataset
for name, df in [("results", results), ("shootouts", shootouts),
                  ("fifa_rank", fifa_rank), ("elo", elo)]:
    print(f"\n--- {name} ---")
    print(df.dtypes)
    print(df.isna().sum()[df.isna().sum() > 0])


In [ ]:
# Cobertura temporal de results.csv
print("Rango de fechas:", results["date"].min(), "a", results["date"].max())
print("Partidos por torneo (top 15):")
print(results["tournament"].value_counts().head(15))
print("\n% de partidos en sede neutral:", round(results["neutral"].mean() * 100, 1), "%")


In [ ]:
# Consistencia de nombres de selección entre fuentes: esto casi siempre revela
# discrepancias (ej. "United States" vs "USA", "IR Iran" vs "Iran", "South Korea"
# vs "Korea Republic"). Se identifican aquí para resolverlas en Data Preparation.
teams_results = set(results["home_team"]).union(results["away_team"])
teams_fifa = set(fifa_rank["team"])
teams_elo = set(elo["team"]) if "team" in elo.columns else set()

print("Selecciones en results.csv sin match directo en fifa_rank:")
print(sorted(teams_results - teams_fifa)[:30])

print("\nSelecciones en fifa_rank sin match directo en results.csv:")
print(sorted(teams_fifa - teams_results)[:30])


## 3. Data Preparation

Pasos aplicados, en orden, y la razón de cada uno:

1. **Normalización de nombres de selección.** Las tres fuentes usan convenciones de nombre distintas
   para el mismo equipo. Sin esto, cualquier `merge` pierde filas silenciosamente.
2. **Integración de tandas de penales.** `shootouts.csv` define quién ganó un partido que terminó
   empatado en tiempo reglamentario (relevante en fases eliminatorias).
3. **Definición del target.** Resultado del partido desde la perspectiva del equipo local
   (`home_win` / `draw` / `away_win`), y diferencia de goles.
4. **Ingeniería de variables de forma reciente**, calculadas de forma *chronológica* (solo con
   información anterior a la fecha del partido) para evitar fuga de información hacia el futuro.
5. **Variables de contexto del partido**: sede neutral, tipo/importancia del torneo.
6. **Unión de fuerza relativa actual (FIFA + Elo)**: solo aplicable como foto del 8 de julio de 2026,
   no por fecha histórica de cada partido. Se documenta como limitación.
7. **Selección final de variables** y exportación de los dos archivos de salida.


### 3.1 Normalización de nombres de selección

Se construye un diccionario de alias manual a partir de las discrepancias detectadas en la sección
anterior. Este diccionario debe revisarse y completarse una vez que los datos reales estén cargados,
ya que el listado exacto de discrepancias depende de la versión descargada de cada fuente.


In [ ]:
# Diccionario de normalización: nombre en la fuente -> nombre estándar del proyecto.
# Se parte de los casos más comunes documentados en la literatura de este dataset de Kaggle;
# se debe ampliar con lo que arroje la celda de comparación de la sección 2 al correr con datos reales.
TEAM_NAME_MAP = {
    "United States": "USA",
    "IR Iran": "Iran",
    "South Korea": "Korea Republic",
    "North Korea": "Korea DPR",
    "Ivory Coast": "Cote d'Ivoire",
    "Cape Verde Islands": "Cape Verde",
    "Congo DR": "DR Congo",
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
}

def normalize_team(name: str) -> str:
    if not isinstance(name, str):
        return name
    name = name.strip()
    name = TEAM_NAME_MAP.get(name, name)
    return name

for df, cols in [(results, ["home_team", "away_team"]),
                  (shootouts, ["home_team", "away_team"]),
                  (fifa_rank, ["team"])]:
    for c in cols:
        df[c] = df[c].apply(normalize_team)

if "team" in elo.columns:
    elo["team"] = elo["team"].apply(normalize_team)

# Re-chequeo rápido de cobertura tras normalizar
teams_results = set(results["home_team"]).union(results["away_team"])
teams_fifa = set(fifa_rank["team"])
print("Selecciones sin match tras normalizar:", sorted(teams_results - teams_fifa)[:30])


### 3.2 Integración de tandas de penales y definición del target


In [ ]:
shootouts_slim = shootouts[["date", "home_team", "away_team", "winner"]].rename(
    columns={"winner": "shootout_winner"}
)

matches = results.merge(
    shootouts_slim, on=["date", "home_team", "away_team"], how="left"
)

matches["decided_by_penalties"] = matches["shootout_winner"].notna()

def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    # Empate en tiempo reglamentario: si hubo penales, el resultado real lo define la tanda.
    if row["decided_by_penalties"]:
        return "home_win" if row["shootout_winner"] == row["home_team"] else "away_win"
    return "draw"

matches["result"] = matches.apply(match_result, axis=1)
matches["goal_diff"] = matches["home_score"] - matches["away_score"]

matches["result"].value_counts(normalize=True).round(3)


### 3.3 Variables de contexto del partido

`neutral` ya viene en el dataset original. Se agrega una variable ordinal simple de importancia del
torneo, porque no es lo mismo un amistoso que un partido de Mundial o de clasificatoria, el nivel de
esfuerzo y de información que aporta el resultado es distinto.


In [ ]:
def tournament_tier(t: str) -> int:
    t = (t or "").lower()
    if "world cup" in t and "qualif" not in t:
        return 4
    if "cup" in t or "championship" in t or "copa" in t or "euro" in t:
        return 3
    if "qualif" in t:
        return 2
    return 1  # amistosos y torneos menores

matches["tournament_tier"] = matches["tournament"].apply(tournament_tier)
matches["neutral"] = matches["neutral"].astype(int)
matches[["tournament", "tournament_tier"]].drop_duplicates().sort_values("tournament_tier", ascending=False).head(10)


### 3.4 Forma reciente por selección (sin fuga de información)

Se transforma `matches` a formato largo (una fila por selección y partido) para poder calcular, por
selección y en orden cronológico, estadísticas de las últimas *N* apariciones **usando `shift` antes
de agregar**, de modo que la fila del partido *t* solo vea información de partidos anteriores a *t*.


In [ ]:
matches = matches.sort_values("date").reset_index(drop=True)
matches["match_id"] = matches.index

home_long = matches[["match_id", "date", "home_team", "home_score", "away_score", "result"]].copy()
home_long.columns = ["match_id", "date", "team", "gf", "ga", "result"]
home_long["is_home"] = 1
home_long["points"] = home_long["result"].map({"home_win": 3, "draw": 1, "away_win": 0})

away_long = matches[["match_id", "date", "away_team", "away_score", "home_score", "result"]].copy()
away_long.columns = ["match_id", "date", "team", "gf", "ga", "result"]
away_long["is_home"] = 0
away_long["points"] = away_long["result"].map({"away_win": 3, "draw": 1, "home_win": 0})

team_long = pd.concat([home_long, away_long], ignore_index=True)
team_long = team_long.sort_values(["team", "date"]).reset_index(drop=True)

WINDOW = 10
grouped = team_long.groupby("team", group_keys=False)

# shift(1) antes de rolling: la fila del partido actual nunca incluye su propio resultado.
team_long["form_points_avg"] = grouped["points"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["form_gf_avg"] = grouped["gf"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["form_ga_avg"] = grouped["ga"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["matches_played_before"] = grouped.cumcount()

team_long[["team", "date", "form_points_avg", "form_gf_avg", "form_ga_avg", "matches_played_before"]].head()


In [ ]:
# Se reconstruye el formato ancho (una fila por partido) uniendo la forma de local y visitante.
form_cols = ["match_id", "team", "form_points_avg", "form_gf_avg", "form_ga_avg", "matches_played_before"]

home_form = team_long.loc[team_long["is_home"] == 1, form_cols].rename(
    columns={c: f"home_{c}" for c in form_cols if c not in ("match_id", "team")}
).drop(columns="team")

away_form = team_long.loc[team_long["is_home"] == 0, form_cols].rename(
    columns={c: f"away_{c}" for c in form_cols if c not in ("match_id", "team")}
).drop(columns="team")

matches = matches.merge(home_form, on="match_id", how="left").merge(away_form, on="match_id", how="left")

# Partidos donde alguno de los dos equipos no tiene historial previo suficiente se excluyen del
# entrenamiento (no del dataset completo) porque su "forma reciente" no es confiable.
matches["has_min_history"] = (matches["home_matches_played_before"] >= 5) & (matches["away_matches_played_before"] >= 5)
matches["has_min_history"].value_counts()


### 3.5 Fuerza relativa actual (ranking FIFA y Elo)

**Limitación importante:** `fifa_rank` y `elo` son una foto tomada el 8 de julio de 2026, no una serie
histórica por fecha. No es correcto unirlas a cada partido histórico como si reflejaran la fuerza del
equipo *en ese momento del pasado*. Por eso se usan en dos lugares distintos:

- En el dataset de entrenamiento (`clean_model_dataset.csv`) **no** se usan como variable directa; el
  modelo aprende de la forma reciente relativa (`home_form_*` vs `away_form_*`) y del contexto del
  partido, que sí son coherentes con la fecha de cada fila.
- En `team_features_2026.csv` sí se incluyen, porque ahí representan correctamente la fuerza *actual*
  de cada selección de cara a la simulación del Mundial 2026.


In [ ]:
fifa_slim = fifa_rank[["team", "rank", "total_points", "confederation"]].rename(
    columns={"rank": "fifa_rank", "total_points": "fifa_points"}
)

if "team" in elo.columns:
    elo_cols = [c for c in ["team", "rank", "rating"] if c in elo.columns]
    elo_slim = elo[elo_cols].rename(columns={"rank": "elo_rank", "rating": "elo_rating"})
else:
    elo_slim = pd.DataFrame(columns=["team", "elo_rank", "elo_rating"])

team_strength = fifa_slim.merge(elo_slim, on="team", how="left")
team_strength.head()


### 3.6 Selección final de variables y exportación

`clean_model_dataset.csv` (nivel partido, para entrenar el modelo de Persona 3):


In [ ]:
model_cols = [
    "match_id", "date", "home_team", "away_team",
    "home_score", "away_score", "goal_diff", "result",
    "neutral", "tournament", "tournament_tier", "decided_by_penalties",
    "home_form_points_avg", "home_form_gf_avg", "home_form_ga_avg", "home_matches_played_before",
    "away_form_points_avg", "away_form_gf_avg", "away_form_ga_avg", "away_matches_played_before",
    "has_min_history",
]

clean_model_dataset = matches[model_cols].copy()
clean_model_dataset.to_csv(PROCESSED_DIR / "clean_model_dataset.csv", index=False)
print("Guardado:", PROCESSED_DIR / "clean_model_dataset.csv", clean_model_dataset.shape)
clean_model_dataset.head()


`team_features_2026.csv` (nivel selección, foto actual para simular el Mundial 2026):

La forma reciente de cada selección se toma de sus últimos `WINDOW` partidos disputados hasta la fecha
de corte de datos (8 de julio de 2026), usando la misma tabla larga `team_long` ya calculada.


In [ ]:
latest_form = (
    team_long.sort_values("date")
    .groupby("team")
    .tail(WINDOW)
    .groupby("team")
    .agg(recent_points_avg=("points", "mean"),
         recent_gf_avg=("gf", "mean"),
         recent_ga_avg=("ga", "mean"),
         last_match_date=("date", "max"))
    .reset_index()
)

team_features_2026 = team_strength.merge(latest_form, on="team", how="left")
team_features_2026.to_csv(PROCESSED_DIR / "team_features_2026.csv", index=False)
print("Guardado:", PROCESSED_DIR / "team_features_2026.csv", team_features_2026.shape)
team_features_2026.sort_values("fifa_rank").head(10)


## 4. Resumen

**Dataset limpio entregado:**

- `data/processed/clean_model_dataset.csv`: un registro por partido histórico, con resultado,
  diferencia de goles, contexto (sede neutral, importancia de torneo) y forma reciente relativa de
  ambos equipos calculada sin fuga de información.
- `data/processed/team_features_2026.csv`: una fila por selección con su ranking FIFA, rating Elo y
  forma reciente actual, para inicializar la simulación del Mundial 2026.

**Variables descartadas y por qué:**

- `goalscorers.csv` y los datasets de Transfermarkt (`opcional/`): no aportan al modelo base y añaden
  complejidad de limpieza (nombres de jugadores, valores de mercado) desproporcionada al tiempo
  disponible. Quedan documentados como fuente opcional.
- Ranking FIFA / Elo histórico por fecha de partido: no disponible (solo tenemos la foto del
  2026-07-08); por eso no se usan como variable de entrenamiento a nivel de partido, solo como
  variable de estado actual para la simulación hacia adelante.

**Pendiente / recomendación para Persona 3:** decidir si el modelo de resultado de partido se entrena
solo con `clean_model_dataset.csv` (histórico) y luego se alimenta con `team_features_2026.csv` para
estimar la fuerza relativa de los cruces del Mundial 2026, o si se combinan ambas fuentes en un único
esquema de features. Ambos archivos comparten la columna `team` con nombres ya normalizados.
